In [91]:
import json
from pathlib import Path
import pandas as pd


def load_experiment_results(base_dir="/home/ossner/git/VoronoiLoss/src/logs"):
    data_list = []
    base_path = Path(base_dir)

    # Use glob to find all matching test_results.json files recursively
    # The pattern matches: logs/{losscombination}/{dataset}/{weight_map}/version_0/eval/version_0/test_results.json
    pattern = "*/*/*/version_0/eval/version_0/test_results.json"

    for json_path in base_path.glob(pattern):
        # Extract variables from the path parts relative to the base directory
        # path.parts looks like: ('logs', 'loss_A', 'dataset_1', 'weight_X', ...)
        # So we can index from the 'logs' directory downwards:
        relative_parts = json_path.relative_to(base_path).parts

        loss_combination = relative_parts[0]
        dataset = relative_parts[1]
        weight_map = relative_parts[2]

        # Read the JSON file
        try:
            with open(json_path, "r") as f:
                metrics = json.load(f)

            # Combine metadata and metrics into one dictionary
            row = {
                "loss_combination": loss_combination,
                "dataset": dataset,
                "weight_map": weight_map,
                **metrics,  # Flattens the JSON content into the dict
            }
            data_list.append(row)

        except (json.JSONDecodeError, IOError) as e:
            print(f"Error reading {json_path}: {e}")

    # Convert the list of dicts into a structured DataFrame
    df = pd.DataFrame(data_list)
    return df


# --- Execution ---
# Generate the dataframe
df = load_experiment_results()

# Optional: Set experiments as a MultiIndex for easier lookup/pivoting
df.set_index(["loss_combination", "dataset", "weight_map"], inplace=True)

In [92]:
df

test/global/dice  \
loss_combination dataset          weight_map                      
110101           plateletclean_cv none                 0.815037   
                 plateletclean_ag none                 0.807099   
                 epflclean_mit    none                 0.943923   
110220           plateletclean_cv none                 0.818509   
                 plateletclean_ag none                 0.808594   
                 epflclean_mit    none                 0.944508   
110110           plateletclean_cv none                 0.810920   
                                  v_mountains          0.800634   
                                  v_iw                 0.731462   
                                  v_region             0.814733   
                                  iw                   0.136071   
                                  v_islands            0.774717   
                                  v_adaptive           0.809028   
                 plateletclean_ag none                 0.820875   
                                  v_mountains          0.796320   
                                  v_iw                 0.716648   
                                  v_region             0.815186   
                                  iw                   0.413048   
                                  v_islands            0.743239   
                                  v_adaptive           0.824181   
                 epflclean_mit    none                 0.943640   
011101           plateletclean_cv none                 0.821598   
                 plateletclean_ag none                 0.812259   
                 epflclean_mit    none                 0.940626   
110000           plateletclean_cv none                 0.804019   
                                  v_mountains          0.807557   
                                  v_iw                 0.771758   
                                  v_region             0.798709   
                                  iw                   0.466966   
                                  v_islands            0.785457   
                                  v_adaptive           0.809456   
                 plateletclean_ag none                 0.812528   
                                  v_mountains          0.800205   
                                  v_iw                 0.754589   
                                  v_region             0.809988   
                                  iw                   0.515002   
                                  v_islands            0.787387   
                                  v_adaptive           0.825175   
                 epflclean_mit    none                 0.943741   
220110           plateletclean_cv none                 0.812297   
                 plateletclean_ag none                 0.812332   
                 epflclean_mit    none                 0.938979   
000110           plateletclean_cv none                 0.813221   
                 plateletclean_ag none                 0.820226   
                 epflclean_mit    none                 0.944656   

                                               test/global/precision  \
loss_combination dataset          weight_map                           
110101           plateletclean_cv none                      0.828842   
                 plateletclean_ag none                      0.802139   
                 epflclean_mit    none                      0.935328   
110220           plateletclean_cv none                      0.849350   
                 plateletclean_ag none                      0.827227   
                 epflclean_mit    none                      0.941203   
110110           plateletclean_cv none                      0.850961   
                                  v_mountains               0.812408   
                                  v_iw                      0.601974   
                                  v_region                  0.841237   
                                 

In [99]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def readable_losses(number_string):
    glob = number_string[:3]
    local = number_string[-3:]
    readablegl = ""
    readablelo = ""
    if glob == '000':
        readablegl += 'none'
    if glob == '220':
        readablegl += '2*DiceCE'
    if glob[0] == '1':
        readablegl += 'Dice'
    if glob[1] == '1':
        readablegl += 'CE'
    if glob[2] == '1':
        readablegl += 'Tversky'
    if local == '000':
        readablelo += 'none'
    if local == '220':
        readablelo += '2*DiceCE'
    if local[0] == '1':
        readablelo += 'Dice'
    if local[1] == '1':
        readablelo += 'CE'
    if local[2] == '1':
        readablelo += 'Tversky'
    return (readablegl, readablelo)

metricreadable = {
  "test/global/dice": "DSC",
  "test/global/F2": "F2",
  "test/instance/f1": "RQ",
  "test/instance/dice": "SQDSC",
  "test/instance/assd": "SQASSD",
  "test/instance/recall":  "inst_recall",
  "test/instance/recall_q0": "inst_recall_q1",
  "test/instance/recall_q1": "inst_recall_q2",
  "test/instance/recall_q2": "inst_recall_q3",
  "test/instance/recall_q3": "inst_recall_q4",
  "test/cc/dice": "CCDice",
  "test/global/precision": "precision",
  "test/global/recall": "recall",
  "test/instance/precision": "inst_precision",
}

# Loss Baseline Comparisons Lollipop Chart

In [100]:
def plot_baseline_comparison_losses(
    df, metric, baseline_tuple, comparison_tuples, save_path
):
    # 1. Safely extract the baseline metric value
    if baseline_tuple not in df.index:
        raise KeyError(
            f"Baseline tuple {baseline_tuple} was not found in the DataFrame index."
        )
    baseline_val = df.loc[baseline_tuple, metric]

    # 2. Extract comparison data and format clean x-axis labels
    comp_vals = []
    labels = []

    for key in comparison_tuples:
        if key in df.index:
            comp_vals.append(df.loc[key, metric])
            # Format label showing loss combination and weight map cleanly on new lines
            labels.append(f"{readable_losses(key[0])[0]}\n{readable_losses(key[0])[1]}")

    comp_vals = np.array(comp_vals)
    x_positions = np.arange(len(labels))

    # 3. Initialize plot layout (using subplots to prevent layout clipping)
    fig, ax = plt.subplots(figsize=(8, 6.5))

    # Draw the horizontal baseline line
    ax.axhline(
        y=baseline_val,
        color="#7f8c8d",
        linestyle="--",
        linewidth=1.8,
        label=f"Baseline (DiceCE, none): {baseline_val:.3f}",
        zorder=1,
    )

    # 4. Generate vertical lines and markers dynamically
    for x, val, label in zip(x_positions, comp_vals, labels):
        if np.isnan(val):
            continue

        diff = val - baseline_val
        # Soft, clean hex colors for presentation: Green for upgrade, Red for downgrade
        if metricreadable[metric] in ['SQASSD', 'CEDI']:
            color = "#616161" if abs(diff) < 0.001 else ("#2ecc71" if diff <= 0 else "#e74c3c")
        else:
            color = "#616161" if abs(diff) < 0.001 else ("#2ecc71" if diff >= 0 else "#e74c3c")

        # Vertical line indicating distance from baseline
        ax.vlines(
            x,
            ymin=baseline_val,
            ymax=val,
            colors=color,
            linewidth=2.5,
            alpha=0.8,
            zorder=2,
        )

        # Dot marker at the performance value
        ax.scatter(x, val, color=color, s=140, zorder=3)

        # Annotate absolute value and relative change
        va = "bottom" if diff >= 0 else "top"
        offset = 0.005 if diff >= 0 else -0.005
        sign = "+" if diff >= 0 else ""

        ax.text(
            x,
            val + offset,
            f"{val:.3f}\n({sign}{diff:.3f})",
            ha="center",
            va=va,
            fontsize=9.5,
            fontweight="bold",
            color=color,
        )

    # 5. Presentation adjustments & Styling
    ax.set_xticks(x_positions)
    ax.set_xmargin(0.1)
    ax.set_xticklabels(labels, rotation=0, fontsize=10, fontweight="medium")
    ax.set_ylabel(metricreadable[metric], fontsize=12, fontweight="bold", labelpad=10)

    # Add dynamic padding to y-axis limits to prevent text labels from clipping
    valid_vals = [v for v in comp_vals if not np.isnan(v)] + [baseline_val]
    ymin, ymax = min(valid_vals), max(valid_vals)
    ax.set_ylim(ymin - 0.04, ymax + 0.04)

    # Soft background grid lines
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    ax.legend(loc="upper left", frameon=True, facecolor="#f8f9fa", edgecolor="#e2e8f0")

    # Save cleanly with tight formatting layout
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    
    
TASK = "mit"
# 1. Define your baseline configuration tuple
baseline_config = ("110000", f"epflclean_{TASK}", "none")

# 2. Provide the list of experimental tuples you wish to compare
comparison_configs = [
    ("000110", f"epflclean_{TASK}", "none"),
    ("110110", f"epflclean_{TASK}", "none"),
    ("220110", f"epflclean_{TASK}", "none"),
    ("110220", f"epflclean_{TASK}", "none"),
    ("110101", f"epflclean_{TASK}", "none"),
    ("011101", f"epflclean_{TASK}", "none"),
]

# 3. Generate and save the visualization
for metric in metricreadable.keys():
    plot_baseline_comparison_losses(
        df=df,
        metric=metric,
        baseline_tuple=baseline_config,
        comparison_tuples=comparison_configs,
        save_path=f'/home/ossner/git/VoronoiLoss/docs/thesis/figures/results/{TASK}/lollipop/loss_combos/{metricreadable[metric]}',
    )

# Weight Map Comparison Lollipop Chart

In [102]:
def plot_baseline_comparison_weightmaps(
    df, metric, baseline_tuple, comparison_tuples, save_path
):
    # 1. Safely extract the baseline metric value
    if baseline_tuple not in df.index:
        raise KeyError(
            f"Baseline tuple {baseline_tuple} was not found in the DataFrame index."
        )
    baseline_val = df.loc[baseline_tuple, metric]

    # 2. Extract comparison data and format clean x-axis labels
    comp_vals = []
    labels = []

    for key in comparison_tuples:
        if key in df.index:
            comp_vals.append(df.loc[key, metric])
            # Format label showing loss combination and weight map cleanly on new lines
            labels.append(f"{key[2]}")

    comp_vals = np.array(comp_vals)
    x_positions = np.arange(len(labels))

    # 3. Initialize plot layout (using subplots to prevent layout clipping)
    fig, ax = plt.subplots(figsize=(8, 6.5))

    # Draw the horizontal baseline line
    ax.axhline(
        y=baseline_val,
        color="#7f8c8d",
        linestyle="--",
        linewidth=1.8,
        label="Baseline ($W_{none}$):" + f"{baseline_val:.3f}",
        zorder=1,
    )

    # 4. Generate vertical lines and markers dynamically
    for x, val, label in zip(x_positions, comp_vals, labels):
        if np.isnan(val):
            continue

        diff = val - baseline_val
        # Soft, clean hex colors for presentation: Green for upgrade, Red for downgrade
        if metricreadable[metric] in ['SQASSD', 'CEDI']:
            color = "#616161" if abs(diff) < 0.001 else ("#2ecc71" if diff <= 0 else "#e74c3c")
        else:
            color = "#616161" if abs(diff) < 0.001 else ("#2ecc71" if diff >= 0 else "#e74c3c")

        # Vertical line indicating distance from baseline
        ax.vlines(
            x,
            ymin=baseline_val,
            ymax=val,
            colors=color,
            linewidth=2.5,
            alpha=0.8,
            zorder=2,
        )

        # Dot marker at the performance value
        ax.scatter(x, val, color=color, s=140, zorder=3)

        # Annotate absolute value and relative change
        va = "bottom" if diff >= 0 else "top"
        offset = 0.005 if diff >= 0 else -0.005
        sign = "+" if diff >= 0 else ""

        ax.text(
            x,
            val + offset,
            f"{val:.3f}\n({sign}{diff:.3f})",
            ha="center",
            va=va,
            fontsize=9.5,
            fontweight="bold",
            color=color,
        )

    # 5. Presentation adjustments & Styling
    ax.set_xticks(x_positions)
    ax.set_xmargin(0.1)
    ax.set_xticklabels(labels, rotation=0, fontsize=10, fontweight="medium")
    ax.set_ylabel(metricreadable[metric], fontsize=12, fontweight="bold", labelpad=10)

    # Add dynamic padding to y-axis limits to prevent text labels from clipping
    valid_vals = [v for v in comp_vals if not np.isnan(v)] + [baseline_val]
    ymin, ymax = min(valid_vals), max(valid_vals)
    ax.set_ylim(ymin - 0.04, ymax + 0.04)

    # Soft background grid lines
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    ax.legend(loc="upper left", frameon=True, facecolor="#f8f9fa", edgecolor="#e2e8f0")

    # Save cleanly with tight formatting layout
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    

TASK = 'mit'
# 1. Define your baseline configuration tuple
baseline_config = ("110000", f"epflclean_{TASK}", "none")

# 2. Provide the list of experimental tuples you wish to compare
comparison_configs = [
    ("110000", f"epflclean_{TASK}", "v_iw"),
    ("110000", f"epflclean_{TASK}", "v_region"),
    ("110000", f"epflclean_{TASK}", "v_adaptive"),
    ("110000", f"epflclean_{TASK}", "v_mountains"),
    ("110000", f"epflclean_{TASK}", "v_islands"),
]

# 3. Generate and save the visualization
for metric in metricreadable.keys():
    plot_baseline_comparison_weightmaps(
        df=df,
        metric=metric,
        baseline_tuple=baseline_config,
        comparison_tuples=comparison_configs,
        save_path=f'/home/ossner/git/VoronoiLoss/docs/thesis/figures/results/{TASK}/lollipop/weight_maps/{metricreadable[metric]}',
    )

# Instance Recall by Quartile - Loss Combos

In [104]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_quartile_baseline_comparison_losses(
    df, baseline_tuple, comparison_tuples, save_path="quartile_comparison_plot.png"
):
    """
    Plots a grouped lollipop chart for 4 volume quartiles (q0, q1, q2, q3)
    relative to their respective baselines.
    """
    quartiles = ["test/instance/recall_q0", "test/instance/recall_q1", "test/instance/recall_q2", "test/instance/recall_q3"]
    quartile_labels = ["Q0", "Q1", "Q2", "Q3"]
    
    # Palette for the four quartiles
    colors = ["#1abc9c", "#3498db", "#9b59b6", "#e67e22"]
    
    if baseline_tuple not in df.index:
        raise KeyError(f"Baseline tuple {baseline_tuple} was not found in the DataFrame index.")
    
    # 1. Extract baselines for all 4 quartiles
    baseline_vals = {q: df.loc[baseline_tuple, q] for q in quartiles}

    # 2. Extract data & build standard labels
    labels = []
    comp_data = {q: [] for q in quartiles}
    
    for key in comparison_tuples:
        if key in df.index:
            labels.append(f"{readable_losses(key[0])[0]}\n{readable_losses(key[0])[1]}")
            for q in quartiles:
                comp_data[q].append(df.loc[key, q])
                
    num_groups = len(labels)
    x_base = np.arange(num_groups)
    
    # 3. Handle Grouping Dimensions
    # total width allowed for all 4 lollipops together per x-tick
    total_group_width = 0.6  
    step = total_group_width / len(quartiles)
    # This centers the group perfectly over the integer x-tick position
    offsets = np.linspace(-total_group_width/2 + step/2, total_group_width/2 - step/2, len(quartiles))

    # 4. Initialize layout (Widened to give 24 lollipops breathing room)
    fig, ax = plt.subplots(figsize=(16, 6.5))

    # Plot baseline dashed lines (shorter, subtle legend tags)
    for idx, q in enumerate(quartiles):
        ax.axhline(
            y=baseline_vals[q],
            color=colors[idx],
            linestyle="--",
            linewidth=1.2,
            alpha=0.6,
            label=f"{quartile_labels[idx]} Base: {baseline_vals[q]:.2f}",
        )

    # 5. Iteratively plot grouped lollipops
    for idx, q in enumerate(quartiles):
        b_val = baseline_vals[q]
        
        for g_idx in range(num_groups):
            val = comp_data[q][g_idx]
            if np.isnan(val):
                continue
                
            # Shift the x position left or right based on which quartile it belongs to
            x_pos = x_base[g_idx] + offsets[idx]
            val = round(val, 3)
            b_val = round(b_val, 3)
            diff = val - b_val
            q_color = "#616161" if abs(diff) < 0.005 else ("#2ecc71" if diff >= 0 else "#e74c3c")
            
            # Draw stem
            ax.vlines(
                x_pos,
                ymin=b_val,
                ymax=val,
                colors=q_color,
                linewidth=2.0,
                alpha=0.8,
                zorder=2,
            )
            
            # Draw marker head
            ax.scatter(x_pos, val, color=q_color, s=70, zorder=3, edgecolors='none')
            
            # Annotation text (smaller size to avoid overcrowding)
            va = "bottom" if diff >= 0 else "top"
            offset = 0.006 if diff >= 0 else -0.006
            sign = "+" if diff >= 0 else ""
            
            ax.text(
                x_pos,
                val + offset,
                f"{sign}{diff:.2f}",
                ha="center",
                va=va,
                fontsize=7.5,
                fontweight="semibold",
                color=q_color,
            )

    # 6. Formatting & Styling
    ax.set_xticks(x_base)
    ax.set_xticklabels(labels, rotation=0, fontsize=10, fontweight="medium")
    ax.set_ylabel("Instance Recall", fontsize=12, fontweight="bold", labelpad=10)
    
    # Push margins slightly so outer lollipops aren't cut off
    ax.set_xmargin(0.08)
    
    # Dynamically find min/max values to safely scale y limits
    all_vals = [v for q in quartiles for v in comp_data[q] if not np.isnan(v)] + list(baseline_vals.values())
    ymin, ymax = min(all_vals), max(all_vals)
    ax.set_ylim(max(0, ymin - 0.06), 1.04)

    ax.grid(axis="y", linestyle=":", alpha=0.5)
    
    # Place Legend cleanly outside or top-left
    ax.legend(loc="upper left", frameon=True, facecolor="#f8f9fa", edgecolor="#e2e8f0", fontsize=9)
    plt.tight_layout()
    #plt.show()
    plt.savefig(save_path, dpi=300)
    plt.close()

# --- Execution ---
TASK = "mit"
# 1. Define your baseline configuration tuple
baseline_config = ("110000", f"epflclean_{TASK}", "none")

# 2. Provide the list of experimental tuples you wish to compare
comparison_configs = [
    ("000110", f"epflclean_{TASK}", "none"),
    ("110110", f"epflclean_{TASK}", "none"),
    ("220110", f"epflclean_{TASK}", "none"),
    ("110220", f"epflclean_{TASK}", "none"),
    ("110101", f"epflclean_{TASK}", "none"),
    ("011101", f"epflclean_{TASK}", "none"),
]

# 3. Generate and save the visualization
plot_quartile_baseline_comparison_losses(
    df=df,
    baseline_tuple=baseline_config,
    comparison_tuples=comparison_configs,
    save_path=f'/home/ossner/git/VoronoiLoss/docs/thesis/figures/results/{TASK}/lollipop/loss_combos/quartile_recall_comparison.png',
)

# Instance Recall by Quartile - Weight Maps

In [82]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_quartile_baseline_comparison_weightmaps(
    df, baseline_tuple, comparison_tuples, save_path="quartile_comparison_plot.png"
):
    """
    Plots a grouped lollipop chart for 4 volume quartiles (q0, q1, q2, q3)
    relative to their respective baselines.
    """
    quartiles = ["test/instance/recall_q0", "test/instance/recall_q1", "test/instance/recall_q2", "test/instance/recall_q3"]
    quartile_labels = ["Q0", "Q1", "Q2", "Q3"]
    
    # Palette for the four quartiles
    colors = ["#1abc9c", "#3498db", "#9b59b6", "#e67e22"]
    
    if baseline_tuple not in df.index:
        raise KeyError(f"Baseline tuple {baseline_tuple} was not found in the DataFrame index.")
    
    # 1. Extract baselines for all 4 quartiles
    baseline_vals = {q: df.loc[baseline_tuple, q] for q in quartiles}

    # 2. Extract data & build standard labels
    labels = []
    comp_data = {q: [] for q in quartiles}
    
    for key in comparison_tuples:
        if key in df.index:
            labels.append(f"{key[2]}")
            for q in quartiles:
                comp_data[q].append(df.loc[key, q])
                
    num_groups = len(labels)
    x_base = np.arange(num_groups)
    
    # 3. Handle Grouping Dimensions
    # total width allowed for all 4 lollipops together per x-tick
    total_group_width = 0.6  
    step = total_group_width / len(quartiles)
    # This centers the group perfectly over the integer x-tick position
    offsets = np.linspace(-total_group_width/2 + step/2, total_group_width/2 - step/2, len(quartiles))

    # 4. Initialize layout (Widened to give 24 lollipops breathing room)
    fig, ax = plt.subplots(figsize=(16, 6.5))

    # Plot baseline dashed lines (shorter, subtle legend tags)
    for idx, q in enumerate(quartiles):
        ax.axhline(
            y=baseline_vals[q],
            color=colors[idx],
            linestyle="--",
            linewidth=1.2,
            alpha=0.6,
            label=f"{quartile_labels[idx]} Base: {baseline_vals[q]:.2f}",
        )

    # 5. Iteratively plot grouped lollipops
    for idx, q in enumerate(quartiles):
        b_val = baseline_vals[q]
        
        for g_idx in range(num_groups):
            val = comp_data[q][g_idx]
            if np.isnan(val):
                continue
                
            # Shift the x position left or right based on which quartile it belongs to
            x_pos = x_base[g_idx] + offsets[idx]
            val = round(val, 3)
            b_val = round(b_val, 3)
            diff = val - b_val
            q_color = "#616161" if abs(diff) < 0.005 else ("#2ecc71" if diff >= 0 else "#e74c3c")
            
            # Draw stem
            ax.vlines(
                x_pos,
                ymin=b_val,
                ymax=val,
                colors=q_color,
                linewidth=2.0,
                alpha=0.8,
                zorder=2,
            )
            
            # Draw marker head
            ax.scatter(x_pos, val, color=q_color, s=70, zorder=3, edgecolors='none')
            
            # Annotation text (smaller size to avoid overcrowding)
            va = "bottom" if diff >= 0 else "top"
            offset = 0.006 if diff >= 0 else -0.006
            sign = "+" if diff >= 0 else ""
            
            ax.text(
                x_pos,
                val + offset,
                f"{sign}{diff:.2f}",
                ha="center",
                va=va,
                fontsize=7.5,
                fontweight="semibold",
                color=q_color,
            )

    # 6. Formatting & Styling
    ax.set_xticks(x_base)
    ax.set_xticklabels(labels, rotation=0, fontsize=10, fontweight="medium")
    ax.set_ylabel("Instance Recall", fontsize=12, fontweight="bold", labelpad=10)
    
    # Push margins slightly so outer lollipops aren't cut off
    ax.set_xmargin(0.08)
    
    # Dynamically find min/max values to safely scale y limits
    all_vals = [v for q in quartiles for v in comp_data[q] if not np.isnan(v)] + list(baseline_vals.values())
    ymin, ymax = min(all_vals), max(all_vals)
    ax.set_ylim(max(0, ymin - 0.06), 1.04)

    ax.grid(axis="y", linestyle=":", alpha=0.5)
    
    # Place Legend cleanly outside or top-left
    ax.legend(loc="upper left", frameon=True, facecolor="#f8f9fa", edgecolor="#e2e8f0", fontsize=9)

    plt.tight_layout()
    #plt.show()
    plt.savefig(save_path, dpi=300)
    plt.close()

# --- Execution ---
TASK = "cv"
# 1. Define your baseline configuration tuple
baseline_config = ("110000", f"plateletclean_{TASK}", "none")

# 2. Provide the list of experimental tuples you wish to compare
comparison_configs = [
    ("110000", f"plateletclean_{TASK}", "v_iw"),
    ("110000", f"plateletclean_{TASK}", "v_region"),
    ("110000", f"plateletclean_{TASK}", "v_adaptive"),
    ("110000", f"plateletclean_{TASK}", "v_mountains"),
    ("110000", f"plateletclean_{TASK}", "v_islands"),
]

# 3. Generate and save the visualization
plot_quartile_baseline_comparison_weightmaps(
    df=df,
    baseline_tuple=baseline_config,
    comparison_tuples=comparison_configs,
    save_path=f'/home/ossner/git/VoronoiLoss/docs/thesis/figures/results/{TASK}/lollipop/weight_maps/quartile_recall_comparison.png',
)

# Metric Comparison Table - Losses

In [89]:
import numpy as np
import pandas as pd


def generate_typst_comparison_table(
    df, metrics_list, metric_readable, baseline_tuple, comparison_tuples
):
    """Generates a Typst table string comparing a baseline configuration

    against experimental combinations across multiple metrics. Includes delta
    color formatting and bolding for the best-performing values.
    """
    # 1. Header setup: Baseline header first, then comparison labels
    headers = [f"[{metric_readable[m]}]" for m in metrics_list]  # Fallback row titles

    # Columns configuration
    num_cols = 1 + 1 + len(comparison_tuples)  # Metric Name + Baseline + Comparisons
    align_str = f"(left, {'center, ' * (num_cols - 1)})".strip(", ")

    typst_lines = [
        "table(",
        f"  columns: (auto, {'1fr, ' * (num_cols - 1)}).slice(0, {num_cols}),",
        f"  align: {align_str},",
        "  stroke: (",
        "    x: none,",
        "    y: 0.4pt + luma(220),",
        "  ),",
        "  inset: 8pt,",
        "",
        "  table.header(",
        "    [],",
    ]

    # Add baseline header (extracting the weight map descriptor name)
    typst_lines.append(f"    [{readable_losses(baseline_tuple[0])}],")

    # Add comparison headers
    for key in comparison_tuples:
        typst_lines.append(f"    [{readable_losses(key[0])}],")
    typst_lines.append("  ),")

    # 2. Process each metric row
    for metric in metrics_list:
        readable_name = metric_readable.get(metric, metric)

        # Handle math formatting variations for specific metrics if needed
        if metric == "F2":
            readable_name = "$F_2$"
        elif metric == "inst_recall":
            readable_name = '$"recall"_"inst"$'

        typst_lines.append(f"\n  // {metric}")
        typst_lines.append(f"  [{readable_name}],")

        # Safely get baseline
        if baseline_tuple not in df.index:
            raise KeyError(f"Baseline {baseline_tuple} not found in DataFrame.")
        baseline_val = df.loc[baseline_tuple, metric]

        # Gather comparison values
        comp_vals = []
        for key in comparison_tuples:
            if key in df.index:
                comp_vals.append(df.loc[key, metric])
            else:
                comp_vals.append(np.nan)

        all_vals = [baseline_val] + comp_vals

        # Determine the "best" score index for bolding
        # Lower is better for SQASSD and CEDI, higher is better for others
        valid_vals = [v for v in all_vals if not np.isnan(v)]
        if not valid_vals:
            continue

        if metric_readable.get(metric, metric) in ["SQASSD", "CEDI"]:
            best_val = min(valid_vals)
        else:
            best_val = max(valid_vals)

        # Render baseline cell
        is_baseline_best = np.isclose(baseline_val, best_val, atol=1e-5)
        b_cell = f"{baseline_val:.3f}"
        if is_baseline_best:
            b_cell = f"*{b_cell}*"
        typst_lines.append(f"  [{b_cell}],")

        # Render comparison cells
        for val in comp_vals:
            if np.isnan(val):
                typst_lines.append("  [-],")
                continue

            diff = val - baseline_val
            is_best = np.isclose(val, best_val, atol=1e-5)

            # Format the delta string
            sign = "+" if diff >= 0 else ""
            delta_cell = f"#delta({sign}{diff:.3f})"

            # Apply bolding wrapper if this cell is the winner
            if is_best:
                delta_cell = f"*{delta_cell}*"

            typst_lines.append(f"  [{delta_cell}],")

    typst_lines.append(")")
    return "\n".join(typst_lines)


# --- Execution Pipeline ---

TASK = "ag"
baseline_config = ("110000", f"plateletclean_{TASK}", "none")
comparison_configs = [
    ("000110", f"plateletclean_{TASK}", "none"),
    ("110110", f"plateletclean_{TASK}", "none"),
    ("220110", f"plateletclean_{TASK}", "none"),
    ("110220", f"plateletclean_{TASK}", "none"),
    ("110101", f"plateletclean_{TASK}", "none"),
    ("011101", f"plateletclean_{TASK}", "none"),
]

# Run the function
typst_table_code = generate_typst_comparison_table(
    df=df,
    metrics_list=metricreadable.keys(),
    metric_readable=metricreadable,
    baseline_tuple=baseline_config,
    comparison_tuples=comparison_configs,
)
print(typst_table_code)

table(
  columns: (auto, 1fr, 1fr, 1fr, 1fr, 1fr, 1fr, 1fr, ).slice(0, 8),
  align: (left, center, center, center, center, center, center, center, ),
  stroke: (
    x: none,
    y: 0.4pt + luma(220),
  ),
  inset: 8pt,

  table.header(
    [],
    [('DiceCE', 'none')],
    [('none', 'DiceCE')],
    [('DiceCE', 'DiceCE')],
    [('2*DiceCE', 'DiceCE')],
    [('DiceCE', '2*DiceCE')],
    [('DiceCE', 'DiceTversky')],
    [('CETversky', 'DiceTversky')],
  ),

  // test/global/dice
  [DSC],
  [0.813],
  [#delta(+0.008)],
  [*#delta(+0.008)*],
  [#delta(-0.000)],
  [#delta(-0.004)],
  [#delta(-0.005)],
  [#delta(-0.000)],

  // test/global/F2
  [F2],
  [0.821],
  [*#delta(+0.030)*],
  [#delta(+0.010)],
  [#delta(+0.003)],
  [#delta(+0.001)],
  [#delta(+0.012)],
  [#delta(+0.025)],

  // test/instance/f1
  [RQ],
  [0.746],
  [#delta(-0.010)],
  [#delta(-0.004)],
  [*#delta(+0.028)*],
  [#delta(+0.000)],
  [#delta(-0.006)],
  [#delta(+0.005)],

  // test/instance/dice
  [SQDSC],
  [0.850],
  [

# Metric Comparison Table - Weight Maps

In [87]:
import numpy as np
import pandas as pd


def generate_typst_comparison_table(
    df, metrics_list, metric_readable, baseline_tuple, comparison_tuples
):
    """Generates a Typst table string comparing a baseline configuration

    against experimental combinations across multiple metrics. Includes delta
    color formatting and bolding for the best-performing values.
    """
    # 1. Header setup: Baseline header first, then comparison labels
    headers = [f"[{metric_readable[m]}]" for m in metrics_list]  # Fallback row titles

    # Columns configuration
    num_cols = 1 + 1 + len(comparison_tuples)  # Metric Name + Baseline + Comparisons
    align_str = f"(left, {'center, ' * (num_cols - 1)})".strip(", ")

    typst_lines = [
        "table(",
        f"  columns: (auto, {'1fr, ' * (num_cols - 1)}).slice(0, {num_cols}),",
        f"  align: {align_str},",
        "  stroke: (",
        "    x: none,",
        "    y: 0.4pt + luma(220),",
        "  ),",
        "  inset: 8pt,",
        "",
        "  table.header(",
        "    [],",
    ]

    # Add baseline header (extracting the weight map descriptor name)
    typst_lines.append(f"    [{baseline_tuple[2]}],")

    # Add comparison headers
    for key in comparison_tuples:
        typst_lines.append(f"    [{key[2]}],")
    typst_lines.append("  ),")

    # 2. Process each metric row
    for metric in metrics_list:
        readable_name = metric_readable.get(metric, metric)

        # Handle math formatting variations for specific metrics if needed
        if metric == "F2":
            readable_name = "$F_2$"
        elif metric == "inst_recall":
            readable_name = '$"recall"_"inst"$'

        typst_lines.append(f"\n  // {metric}")
        typst_lines.append(f"  [{readable_name}],")

        # Safely get baseline
        if baseline_tuple not in df.index:
            raise KeyError(f"Baseline {baseline_tuple} not found in DataFrame.")
        baseline_val = df.loc[baseline_tuple, metric]

        # Gather comparison values
        comp_vals = []
        for key in comparison_tuples:
            if key in df.index:
                comp_vals.append(df.loc[key, metric])
            else:
                comp_vals.append(np.nan)

        all_vals = [baseline_val] + comp_vals

        # Determine the "best" score index for bolding
        # Lower is better for SQASSD and CEDI, higher is better for others
        valid_vals = [v for v in all_vals if not np.isnan(v)]
        if not valid_vals:
            continue

        if metric_readable.get(metric, metric) in ["SQASSD", "CEDI"]:
            best_val = min(valid_vals)
        else:
            best_val = max(valid_vals)

        # Render baseline cell
        is_baseline_best = np.isclose(baseline_val, best_val, atol=1e-5)
        b_cell = f"{baseline_val:.3f}"
        if is_baseline_best:
            b_cell = f"*{b_cell}*"
        typst_lines.append(f"  [{b_cell}],")

        # Render comparison cells
        for val in comp_vals:
            if np.isnan(val):
                typst_lines.append("  [-],")
                continue

            diff = val - baseline_val
            is_best = np.isclose(val, best_val, atol=1e-5)

            # Format the delta string
            sign = "+" if diff >= 0 else ""
            delta_cell = f"#delta({sign}{diff:.3f})"

            # Apply bolding wrapper if this cell is the winner
            if is_best:
                delta_cell = f"*{delta_cell}*"

            typst_lines.append(f"  [{delta_cell}],")

    typst_lines.append(")")
    return "\n".join(typst_lines)


# --- Execution Pipeline ---

TASK = "cv"
baseline_config = ("110000", f"plateletclean_{TASK}", "none")
comparison_configs = [
    ("110000", f"plateletclean_{TASK}", "v_iw"),
    ("110000", f"plateletclean_{TASK}", "v_region"),
    ("110000", f"plateletclean_{TASK}", "v_adaptive"),
    ("110000", f"plateletclean_{TASK}", "v_mountains"),
    ("110000", f"plateletclean_{TASK}", "v_islands"),
]

# Run the function
typst_table_code = generate_typst_comparison_table(
    df=df,
    metrics_list=metricreadable.keys(),
    metric_readable=metricreadable,
    baseline_tuple=baseline_config,
    comparison_tuples=comparison_configs,
)
print(typst_table_code)

table(
  columns: (auto, 1fr, 1fr, 1fr, 1fr, 1fr, 1fr, ).slice(0, 7),
  align: (left, center, center, center, center, center, center, ),
  stroke: (
    x: none,
    y: 0.4pt + luma(220),
  ),
  inset: 8pt,

  table.header(
    [],
    [none],
    [v_iw],
    [v_region],
    [v_adaptive],
    [v_mountains],
    [v_islands],
  ),

  // test/global/dice
  [DSC],
  [0.804],
  [#delta(-0.032)],
  [#delta(-0.005)],
  [*#delta(+0.005)*],
  [#delta(+0.004)],
  [#delta(-0.019)],

  // test/global/F2
  [F2],
  [0.777],
  [*#delta(+0.075)*],
  [#delta(-0.009)],
  [#delta(+0.007)],
  [#delta(+0.014)],
  [#delta(+0.067)],

  // test/instance/f1
  [RQ],
  [*0.875*],
  [#delta(-0.081)],
  [#delta(-0.005)],
  [#delta(-0.002)],
  [#delta(-0.030)],
  [#delta(-0.031)],

  // test/instance/dice
  [SQDSC],
  [0.812],
  [#delta(-0.033)],
  [#delta(-0.003)],
  [#delta(+0.004)],
  [*#delta(+0.007)*],
  [#delta(-0.024)],

  // test/instance/assd
  [SQASSD],
  [0.392],
  [#delta(+0.201)],
  [#delta(+0.011)],
 